In [6]:
import mlflow
from pathlib import Path

from src.build_spark import spark_session_context

from src.training import predict_rul
from src.tracking import fetch_run_id
from src.data_processing import load_cmapss_raw

In [7]:
project_root = Path("/home/aanchal/nasa_c_mapss")

mlflow.set_tracking_uri("file://" + str(project_root / "mlruns"))

In [8]:
print(Path.cwd())
print(mlflow.get_tracking_uri())

/home/aanchal/nasa_c_mapss/code
file:///home/aanchal/nasa_c_mapss/mlruns


In [9]:
subset_id = "FD004"
unit_id = "5"

In [15]:
with spark_session_context(app_name="cmapss-rul-estimation") as spark:

    test_data = load_cmapss_raw(spark, project_root / "Data/CMAPSSData/test_FD004.txt")    

    raw_trajectory = (test_data.where(f"unit_id = {unit_id}").orderBy("cycle"))    
    raw_trajectory.show()

    prediction = predict_rul(subset_id=subset_id, 
                             training_run_id=fetch_run_id(subset_id, experiment_type="training", model_type="RandomForestRegressor"), 
                             raw_trajectory=raw_trajectory)

print(f"Predicted RUL: {prediction:.2f}")

+-------+-----+---------+---------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+
|unit_id|cycle|setting_1|setting_2|setting_3|sensor_1|sensor_2|sensor_3|sensor_4|sensor_5|sensor_6|sensor_7|sensor_8|sensor_9|sensor_10|sensor_11|sensor_12|sensor_13|sensor_14|sensor_15|sensor_16|sensor_17|sensor_18|sensor_19|sensor_20|sensor_21|
+-------+-----+---------+---------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+
|      5|    1|  42.0003|     0.84|    100.0|   445.0|  549.73| 1352.04| 1123.66|    3.91|    5.71|  138.31| 2211.86| 8323.79|     1.02|     42.3|   130.54|  2387.91|  8082.97|   9.3695|     0.02|    331.0|   2212.0|    100.0|    10.59|   6.3617|
|      5|   

Predicted RUL: 68.03


# Server Check

In [1]:
import json
from urllib.request import Request, urlopen

In [2]:
BASE_URL = "http://localhost:8000"

In [3]:
def call_api(path, payload=None):
    body = json.dumps(payload).encode() if payload is not None else None
    request = Request(
        f"{BASE_URL}{path}",
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST" if payload is not None else "GET",
    )

    with urlopen(request) as response:
        return json.load(response)

In [4]:
print(call_api("/health"))

{'status': 'ok', 'spark_ready': True}


In [5]:
observation = {
    "unit_id": 1,
    "cycle": 1,
    "setting_1": 0.0,
    "setting_2": 0.0,
    "setting_3": 100.0,
    **{
        f"sensor_{number}": 0.0
        for number in range(1, 22)
    },
}

payload = {
    "subset_id": "FD004",
    "model_type": "RandomForestRegressor",
    "observations": [observation],
}

prediction = call_api("/predict", payload)
print(prediction)
print("Predicted RUL:", prediction["predicted_rul"])

{'subset_id': 'FD004', 'model_type': 'RandomForestRegressor', 'predicted_rul': 254.13}
Predicted RUL: 254.13
